# Python for Poets 2026 - Workbook 4B - Topics in Parliament

In Workbook 4A we compared groups of speeches one word at a time. A **topic model** looks for groups of words that tend to turn up *together*, in the same speeches: "school", "teachers", "pupils"; or "police", "crime", "officers". Each group is a **topic**. Nobody tells the model what the topics are, or even what a topic is: it finds them from patterns of words alone. Deciding what they mean is our job.

## Setup

In [ ]:
#@title Install and import the `python4poets` package
try:
    import python4poets as p4p
except ImportError:
    # Not installed yet: install it into this notebook's Python, then import
    %pip install -q git+https://github.com/quadrismegistus/python-for-poets
    import python4poets as p4p

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#@title Download and open the Hansard sample (as in Workbook 4A)
df = p4p.load_hansard()
df

## What the model sees

A topic model doesn't read. For each speech, all it sees is which words it uses, and how often: a "bag of words". It also leaves out:

- stopwords ("the", "of", "and"), which every speech uses;
- words used in fewer than 20 speeches, which are too rare to tell us about patterns;
- words used in more than half of all speeches, which are too common to tell speeches apart.

In [ ]:
#@title One speech, as the model sees it
speech = df.loc[0, 'txt']
print(speech)
print()
p4p.count_words(speech, stopwords='english').most_common()

∇ *What is lost when a speech becomes a bag of words? What survives? Could you guess what this speech is about from the counts alone?*

A topic model treats every speech as a *mixture* of a small number of topics (say, 60% schools, 30% money, 10% procedure), and every topic as a list of words, each with a weight. It then looks for the topics, and the mixtures, that best reproduce the word counts of all the speeches at once. Words that keep turning up in the same speeches end up in the same topic.

There are several ways to do this. The classic one in the digital humanities is **LDA** (Latent Dirichlet Allocation). It imagines that every speech was written like this: the speaker picks a few topics, in some proportions, and then picks each word by first picking one of those topics and then a word typical of it. That's not how anyone writes, of course. But if we *pretend* it is, we can work backwards from the speeches to the topics and proportions that would most likely have produced them.

We'll mostly use a faster method, **NMF** (Non-negative Matrix Factorization), which on short texts like these speeches tends to give clearer topics. It has no story about how speeches were written: it simply looks for the set of topics whose mixtures come closest to each speech's *weighted* word counts. Later on we'll compare the two.

### Weighting words: tf-idf

NMF doesn't use raw word counts. First, each count is reweighted by **tf-idf**, short for *term frequency × inverse document frequency*:

- **term frequency** (tf): how often the word is used in this speech;
- **inverse document frequency** (idf): how *rare* the word is across the whole corpus. It is the logarithm of (the number of speeches ÷ the number of speeches that use the word): a word used in nearly every speech gets an idf near 0, a word used in one speech in a thousand gets a high one.

Multiplied together, a word scores highly in a speech if this speech uses it a lot *and* few other speeches do. That makes it a good clue to what this speech, in particular, is about.

The idea of judging what a document is about by counting its words goes back to Hans Peter Luhn at IBM in the 1950s. Inverse document frequency was invented by **Karen Spärck Jones**, in a 1972 paper on how to rank documents in a search ("A statistical interpretation of term specificity and its application in retrieval"). She worked here in Cambridge from the late 1950s: first at the Cambridge Language Research Unit, an independent lab founded by Margaret Masterman that pioneered computational linguistics and machine translation, and later at the University's Computer Laboratory. Versions of her idea are still at the heart of search engines today.

In [ ]:
#@title Inverse document frequency for a few words
import math

n = len(df)
for word in ['government', 'people', 'school', 'nhs', 'fisheries', 'spitalfields']:
    n_texts = df['txt'].str.contains(rf'\b{word}\b', case=False).sum()
    print(f'{word}: in {n_texts:,} of {n:,} speeches; idf = log({n:,} / {n_texts:,}) = {math.log(n / n_texts):.2f}')

∇ *Which of these words would help most to tell speeches apart? Which word has the highest idf, and would you want it in a topic? (Remember "Spitalfields" from Workbook 4A.)*

Very rare words get the highest idf of all, which is one reason the model leaves out words used in fewer than 20 speeches. At the other end, words used in more than half of all speeches are left out altogether; "government", in about two speeches in five, stays in, but with a low weight.

## Finding topics

In [ ]:
#@title Find 30 topics (this takes a minute or so)
model = p4p.topic_model(df, n_topics=30)
model

The result is a `TopicModel`. Type `model.` in a new cell and press Tab to see what it can do.

In [ ]:
#@title The topics
model.topics

For each topic, `words` lists the words most typical of it, and `share` is how much of an average speech it accounts for. (The shares add up to 1.)

∇ *Go down the list and give each topic a name in a word or two. Which topics are about a **subject** (health, Northern Ireland...)? Which are about something else, like the **procedure** of Parliament, or a **way of speaking**? Are there any you can't name?*

The words alone can mislead. To understand a topic, read the speeches that have the most of it.

In [ ]:
#@title Find the topic that the word "nhs" belongs to most, and look at its top words
health = model.find_topic('nhs')
model.top_words(health)

In [ ]:
#@title The speeches with the most of this topic
model.top_texts(health)

∇ *Read two or three of these speeches (click on a cell in the `txt` column, or use the interactive table). Does "health" fit? What would be a better name?*

In [ ]:
#@title TODO: Pick a topic whose words puzzle you, and read its top speeches
topic = REPLACE_ME
print(model.topics.loc[topic, 'words'])
model.top_texts(topic)

Once you've decided what a topic is, give it a name; it will be used in tables and plots from now on.

In [ ]:
#@title Name some topics
model.name_topic(health, 'health')
model.topics

In [ ]:
#@title TODO: Name at least three more topics
model.name_topic(REPLACE_ME, 'REPLACE_ME_ALSO')

## One speech, many topics

Every speech is a *mixture* of topics.

In [ ]:
#@title Which topics is the first speech made of?
model.text_topics(0)

In [ ]:
#@title TODO: Pick another speech (by its row number) and look at its topics; then read it
row = REPLACE_ME
print(df.loc[row, 'speaker'], df.loc[row, 'date'])
print(df.loc[row, 'txt'])
model.text_topics(row)

∇ *Does the mixture match your sense of what the speech is about? Which topics are "about" the speech's subject, and which about how it is said?*

## Topics and metadata

Now we can ask the questions of Workbook 4A about topics instead of words. `model.texts` is our DataFrame with one extra column per topic: each speech's share of that topic.

In [ ]:
#@title Each speech's share of each topic
model.texts

In [ ]:
#@title Each topic's average share of speeches, by party
model.topics_by_group('party_main')

As with words, we can look for the topics most distinctive of one group. Here the test compares *speeches* rather than words: is a topic's share generally higher in one group's speeches than the other's? So one MP who talks about one thing a great deal can't swing the result, the problem we met in Workbook 4A.

In [ ]:
#@title Distinctive topics: Conservative vs. Labour
result = model.distinctive_topics('party_main', 'Conservative', 'Labour')
p4p.plot_distinctive_words(result)

∇ *Which topics belong to each party? Remember "party, or position?" from Workbook 4A: which of these differences might really be about government vs. opposition?*

In [ ]:
#@title TODO: Distinctive topics: government vs. opposition
result_side = model.distinctive_topics('REPLACE_ME', 'government', 'opposition')
p4p.plot_distinctive_words(result_side)

∇ *Look at the `p` and `p_adj` columns. With 40,000 speeches on each side, almost every difference is "significant", however small. Which column tells you whether a difference is big enough to matter?*

As in Workbook 4A, women's speeches are concentrated in recent years, so to compare women and men we should hold the period constant. `rows=` picks which speeches to compare:

In [ ]:
#@title TODO: Distinctive topics: women vs. men, since 2010
result_gender = model.distinctive_topics('gender', 'female', 'male', rows=model.texts['year'] >= REPLACE_ME)
p4p.plot_distinctive_words(result_gender)

∇ *Compare these topics with the distinctive words for women and men since 2010 in Workbook 4A. Do the two methods tell the same story?*

### Topics over time

In [ ]:
#@title The health topic over time, by party
model.plot_over_time(health, color='party_main')

In [ ]:
#@title Several topics over time
europe = model.find_topic('european')
ireland = model.find_topic('ireland')
model.plot_over_time([health, europe, ireland])

In [ ]:
#@title TODO: Plot a topic of your choice over time, colored by party or side
model.plot_over_time(REPLACE_ME, color='REPLACE_ME_ALSO')

∇ *Pick a rise or fall in one of your plots. Read some of the topic's top speeches from before and after: what changed?*

## How much should we trust a topic?

A topic model is not a measurement of the corpus; it's one of many possible summaries of it. The topics depend on choices we made (which algorithm, how many topics, which words to leave out) and, for LDA, even on chance: it starts from random guesses and improves them. Topics that come back whatever the choices are more trustworthy than ones that don't.

To save time, let's experiment on a smaller sample of 20,000 speeches, starting with LDA.

In [ ]:
#@title Two LDA models of the same speeches, from different random starting points
df_small = df.sample(20_000, random_state=1)

lda_a = p4p.topic_model(df_small, n_topics=30, method='lda', seed=1)
lda_b = p4p.topic_model(df_small, n_topics=30, method='lda', seed=2)

lda_a.match(lda_b)

`match` finds, for each topic in `lda_a`, the most similar topic in `lda_b`, and how similar they are (1 = the same words with the same weights; 0 = no words in common).

∇ *Which topics come back almost unchanged? Which don't? What kinds of topics are the stable ones?*

In [ ]:
#@title The same speeches, with NMF
nmf_small = p4p.topic_model(df_small, n_topics=30)
nmf_small.topics

In [ ]:
#@title Which LDA topics does NMF also find?
lda_a.match(nmf_small)

∇ *Which topics do both methods find? Remember that the two don't even see the same input: LDA works on raw word counts, NMF on tf-idf weights. How might that explain some of the differences?*

In [ ]:
#@title TODO: Fewer or more topics: try 10, then 60
model_k = p4p.topic_model(df_small, n_topics=REPLACE_ME)
model_k.topics

∇ *With 10 topics, which of the 30 topics (in `nmf_small`) get merged together? With 60, which get split apart? Is there a "right" number of topics?*

Finally, what counts as noise is also a choice. Many topics are full of the language of the Commons itself ("hon", "friend", "gentleman", "minister"). We can leave those words out, as extra stopwords:

In [ ]:
#@title TODO: Leave out the language of procedure
model_no_procedure = p4p.topic_model(
    df_small, n_topics=30,
    extra_stopwords=['hon', 'friend', 'gentleman', 'lady', 'right', 'member', 'members', 'mr', 'speaker', 'REPLACE_ME'],
)
model_no_procedure.topics

∇ *What happened to the procedural topics? Are the subject topics clearer now? What did we lose? (Remember "my hon. Friend" in Workbook 4A: was it noise?)*

## Your turn

In [ ]:
#@title TODO: Pick one topic. Name it, read its top speeches, plot it over time, and compare it between two groups. Then write a few sentences on what you found, and how much you trust it